In [1]:
# Week 3 Day 2 — Tensor Properties
# Goal: Master shape, dtype, device

import torch
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✓ Setup complete")

Device: cuda
✓ Setup complete


In [3]:
# Cell 2 — Shape manipulation

# Start with a simple tensor
x = torch.zeros(4, 512, 512)
print(f"Original shape: {x.shape}")
print(f"Dimensions:     {x.ndim}")
print(f"Total elements: {x.numel()}")

print("\n── reshape ──")
# reshape changes dimensions
# total elements must stay same
# 4 × 512 × 512 = 1,048,576 elements
x_reshaped = x.reshape(2, 2, 512, 512)
print(f"After reshape(2,2,512,512): {x_reshaped.shape}")

print("\n── unsqueeze ──")
# adds a new dimension
x = torch.zeros(512, 512)
print(f"Before unsqueeze: {x.shape}")

x = x.unsqueeze(0)  # add at position 0
print(f"After unsqueeze(0): {x.shape}")

x = x.unsqueeze(0)  # add again at position 0
print(f"After unsqueeze(0): {x.shape}")

print("\n── squeeze ──")
# removes dimensions of size 1
# opposite of unsqueeze
print(f"Before squeeze: {x.shape}")
x = x.squeeze()
print(f"After squeeze:  {x.shape}")

print("\n── permute ──")
# rearranges dimensions
# useful when data comes in wrong order
x = torch.zeros(512, 512, 1)
print(f"Before permute: {x.shape}")
# move channel from last to first position
x = x.permute(2, 0, 1)
print(f"After permute:  {x.shape}")

Original shape: torch.Size([4, 512, 512])
Dimensions:     3
Total elements: 1048576

── reshape ──
After reshape(2,2,512,512): torch.Size([2, 2, 512, 512])

── unsqueeze ──
Before unsqueeze: torch.Size([512, 512])
After unsqueeze(0): torch.Size([1, 512, 512])
After unsqueeze(0): torch.Size([1, 1, 512, 512])

── squeeze ──
Before squeeze: torch.Size([1, 1, 512, 512])
After squeeze:  torch.Size([512, 512])

── permute ──
Before permute: torch.Size([512, 512, 1])
After permute:  torch.Size([1, 512, 512])


In [4]:
# Cell 3 — dtype in practice

print("── Common dtypes ──")

# Integer types
int_tensor = torch.tensor([100, 200, 300])
print(f"Default integer: {int_tensor.dtype}")

# Float types
float_tensor = torch.tensor([100.0, 200.0, 300.0])
print(f"Default float:   {float_tensor.dtype}")

print("\n── Conversions ──")

# CT image → must be float32
ct = torch.tensor([-1000, 42, 400])
print(f"CT before: {ct.dtype}  values: {ct}")
ct = ct.float()
print(f"CT after:  {ct.dtype}  values: {ct}")

# Segmentation mask → stays int64
mask = torch.tensor([0, 1, 0, 1])
print(f"\nMask: {mask.dtype}  values: {mask}")

# Model weights → always float32
weights = torch.randn(3, 3)
print(f"\nWeights dtype: {weights.dtype}")
print(f"Weights:\n{weights}")

print("\n── dtype rule ──")
print("Images  → float32")
print("Labels  → int64")
print("Weights → float32")

── Common dtypes ──
Default integer: torch.int64
Default float:   torch.float32

── Conversions ──
CT before: torch.int64  values: tensor([-1000,    42,   400])
CT after:  torch.float32  values: tensor([-1000.,    42.,   400.])

Mask: torch.int64  values: tensor([0, 1, 0, 1])

Weights dtype: torch.float32
Weights:
tensor([[-0.5552, -0.8320,  0.2237],
        [ 1.5732, -1.2266, -0.0445],
        [-1.4428,  0.2769,  0.0275]])

── dtype rule ──
Images  → float32
Labels  → int64
Weights → float32


In [5]:
# Cell 4 — Device management

print("── Device setup ──")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Available device: {device}")

print("\n── Creating tensors ──")
# Tensor starts on CPU by default
ct = torch.zeros(512, 512)
print(f"Default device: {ct.device}")

# Move to GPU
ct = ct.to(device)
print(f"After .to(device): {ct.device}")

print("\n── Device mismatch error ──")
# Model on GPU, data on CPU → ERROR
# Let us demonstrate

tensor_gpu = torch.zeros(3).to(device)
tensor_cpu = torch.zeros(3)

print(f"tensor_gpu device: {tensor_gpu.device}")
print(f"tensor_cpu device: {tensor_cpu.device}")

try:
    result = tensor_gpu + tensor_cpu
except Exception as e:
    print(f"\nError: {e}")

print("\n── Fix the error ──")
tensor_cpu = tensor_cpu.to(device)
print(f"tensor_cpu after fix: {tensor_cpu.device}")
result = tensor_gpu + tensor_cpu
print(f"Addition successful: {result}")

print("\n── Device rule ──")
print("Data and model must be on same device")
print("Always use .to(device) for both")

── Device setup ──
Available device: cuda

── Creating tensors ──
Default device: cpu
After .to(device): cuda:0

── Device mismatch error ──
tensor_gpu device: cuda:0
tensor_cpu device: cpu

Error: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

── Fix the error ──
tensor_cpu after fix: cuda:0
Addition successful: tensor([0., 0., 0.], device='cuda:0')

── Device rule ──
Data and model must be on same device
Always use .to(device) for both


In [6]:
# Cell 5 — Real CT example
# All three properties together

import numpy as np

print("── Loading CT data ──")
# Simulate loading CT with pydicom
ct_numpy = np.random.randint(-1000, 400,
                              size=(512, 512),
                              dtype=np.int16)
mask_numpy = np.random.randint(0, 2,
                                size=(512, 512),
                                dtype=np.int64)

print(f"CT numpy   → shape: {ct_numpy.shape}  dtype: {ct_numpy.dtype}")
print(f"Mask numpy → shape: {mask_numpy.shape}  dtype: {mask_numpy.dtype}")

print("\n── Fixing dtype ──")
# CT → float32
ct_tensor = torch.from_numpy(ct_numpy.astype(np.float32))
# Mask → int64 (already correct)
mask_tensor = torch.from_numpy(mask_numpy)

print(f"CT tensor   → dtype: {ct_tensor.dtype}")
print(f"Mask tensor → dtype: {mask_tensor.dtype}")

print("\n── Fixing device ──")
ct_tensor   = ct_tensor.to(device)
mask_tensor = mask_tensor.to(device)

print(f"CT device:   {ct_tensor.device}")
print(f"Mask device: {mask_tensor.device}")

print("\n── Fixing shape ──")
# Add channel and batch dimensions
ct_tensor   = ct_tensor.unsqueeze(0).unsqueeze(0)
mask_tensor = mask_tensor.unsqueeze(0).unsqueeze(0)

print(f"CT shape:   {ct_tensor.shape}")
print(f"Mask shape: {mask_tensor.shape}")

print("\n── Normalise CT ──")
ct_tensor = (ct_tensor - (-1000.0)) / (400.0 - (-1000.0))
print(f"CT range: {ct_tensor.min():.3f} to {ct_tensor.max():.3f}")

print("\n── Final check ──")
print(f"CT   → shape: {ct_tensor.shape}  dtype: {ct_tensor.dtype}  device: {ct_tensor.device}")
print(f"Mask → shape: {mask_tensor.shape}  dtype: {mask_tensor.dtype}  device: {mask_tensor.device}")
print("\n✓ CT and mask ready for U-Net training")

── Loading CT data ──
CT numpy   → shape: (512, 512)  dtype: int16
Mask numpy → shape: (512, 512)  dtype: int64

── Fixing dtype ──
CT tensor   → dtype: torch.float32
Mask tensor → dtype: torch.int64

── Fixing device ──
CT device:   cuda:0
Mask device: cuda:0

── Fixing shape ──
CT shape:   torch.Size([1, 1, 512, 512])
Mask shape: torch.Size([1, 1, 512, 512])

── Normalise CT ──
CT range: 0.000 to 0.999

── Final check ──
CT   → shape: torch.Size([1, 1, 512, 512])  dtype: torch.float32  device: cuda:0
Mask → shape: torch.Size([1, 1, 512, 512])  dtype: torch.int64  device: cuda:0

✓ CT and mask ready for U-Net training
